# Local PR2 in a simple MuJoCo world

This reproduces the lightweight apartment-style test world and uses `semantic_digital_twin/resources/urdf/pr2_kinematic_tree.urdf`. The full apartment is not used because one textured Collada mesh currently fails in the MuJoCo converter.

In [ ]:
import os
from pathlib import Path
from semantic_digital_twin.adapters.multi_sim import MujocoSim
from semantic_digital_twin.adapters.urdf import URDFParser
from semantic_digital_twin.datastructures.prefixed_name import PrefixedName
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.spatial_types.spatial_types import HomogeneousTransformationMatrix
from semantic_digital_twin.world_description.connections import Connection6DoF, FixedConnection, OmniDrive
from semantic_digital_twin.world_description.geometry import Box, Color, Scale
from semantic_digital_twin.world_description.shape_collection import ShapeCollection
from semantic_digital_twin.world_description.world_entity import Body

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'semantic_digital_twin/resources/urdf/pr2_kinematic_tree.urdf').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the repository.')

REPO_ROOT = find_repo_root()
PR2_URDF = REPO_ROOT / 'semantic_digital_twin/resources/urdf/pr2_kinematic_tree.urdf'
HEADLESS = os.environ.get('MUJOCO_HEADLESS', '0') == '1'
print(REPO_ROOT, PR2_URDF, f'headless={HEADLESS}', sep='\n')

## Build our PR2

The local URDF omits decorative links referenced by `pr2.srdf`, so this initializes the repository PR2 semantics while skipping only those incompatible collision-ignore rules.

In [ ]:
world = URDFParser.from_file(file_path=str(PR2_URDF)).parse()
with world.modify_world():
    pr2_root = world.root
    map_body = Body(name=PrefixedName('map'))
    odom_body = Body(name=PrefixedName('odom_combined'))
    map_to_odom = Connection6DoF.create_with_dofs(world=world, parent=map_body, child=odom_body)
    world.add_connection(map_to_odom)
    drive = OmniDrive.create_with_dofs(world=world, parent=odom_body, child=pr2_root)
    world.add_connection(drive)
    drive.has_hardware_interface = True
    pr2 = PR2._init_empty_robot(world)
    pr2._setup_semantic_annotations()
    world.add_semantic_annotation(pr2)
    pr2._setup_velocity_limits()
    pr2._setup_hardware_interfaces()
    pr2._setup_joint_states()
print(f'{len(world.bodies)} PR2 bodies, {len(pr2.arms)} arms')

## Add the simple apartment-style test scene

In [ ]:
def make_box(name, size, color):
    body = Body(name=PrefixedName(name))
    body.visual = ShapeCollection([Box(scale=Scale(*size), color=Color(*color))], reference_frame=body)
    body.collision = ShapeCollection([Box(scale=Scale(*size), color=Color(*color))], reference_frame=body)
    return body

def add_fixed_box(name, size, xyz, color):
    body = make_box(name, size, color)
    world.add_connection(FixedConnection(
        parent=world.root, child=body,
        parent_T_connection_expression=HomogeneousTransformationMatrix.from_xyz_rpy(*xyz, reference_frame=world.root)))
    return body

with world.modify_world():
    floor = add_fixed_box('floor', (6.0, 6.0, 0.1), (0.0, 0.0, -0.05), (0.65, 0.65, 0.65, 1.0))
    table = add_fixed_box('table', (1.2, 0.8, 0.75), (1.2, 0.0, 0.375), (0.45, 0.25, 0.10, 1.0))
    add_fixed_box('wall_back', (6.0, 0.1, 2.0), (0.0, 3.0, 1.0), (0.8, 0.8, 0.85, 1.0))
    add_fixed_box('wall_left', (0.1, 6.0, 2.0), (-3.0, 0.0, 1.0), (0.8, 0.8, 0.85, 1.0))
    pickup_cube = make_box('pickup_cube', (0.08, 0.08, 0.16), (0.9, 0.1, 0.1, 1.0))
    cube_connection = Connection6DoF.create_with_dofs(world=world, parent=world.root, child=pickup_cube)
    world.add_connection(cube_connection)

drive.origin = HomogeneousTransformationMatrix.from_xyz_rpy(reference_frame=odom_body)
cube_connection.origin = HomogeneousTransformationMatrix.from_xyz_rpy(x=0.85, y=0.0, z=0.83, reference_frame=world.root)
world.notify_state_change()
print(f'{len(world.bodies)} bodies, {len(world.connections)} connections')

## Launch MuJoCo

A viewer window opens unless `MUJOCO_HEADLESS=1`. Gravity is disabled because SDT controls this PR2 kinematically.

In [ ]:
mujoco_sim = MujocoSim(world=world, headless=HEADLESS, gravity=False, contact=True)
mujoco_sim.start_simulation()
print('MuJoCo started.')

## Optional base movement

In [ ]:
drive.origin = HomogeneousTransformationMatrix.from_xyz_rpy(x=0.25, y=-0.25, yaw=0.25, reference_frame=odom_body)
world.notify_state_change()
print('PR2 base moved.')

## Stop before rerunning the notebook

In [ ]:
if 'mujoco_sim' in globals() and mujoco_sim.is_running():
    mujoco_sim.stop_simulation()
    print('MuJoCo stopped.')